# AgroVision Cassava — Competition Training Pipeline

A reproducible Google Colab pipeline for the official **Cassava Leaf Disease Classification** Kaggle competition dataset (21,397 field images, five classes).

This notebook performs dataset integrity checks, exact-duplicate removal, a frozen stratified split, two-stage EfficientNetV2 training, early stopping, class weighting, calibration, selective prediction, confusion matrices, bootstrap confidence intervals, and model export.

> **Claim boundary:** The final test split is an internal held-out African-dataset evaluation. It is not an external Thai field validation. Keep a separately collected, expert-labelled Thai cohort untouched for the final deployment claim.

## 0. Colab preparation
Select **Runtime → Change runtime type → T4 GPU** before running. Run cells from top to bottom once.

In [ ]:
!pip -q install --upgrade kaggle

import os, json, random, hashlib, shutil, subprocess, sys, time
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, balanced_accuracy_score, f1_score
from scipy.optimize import minimize_scalar

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

gpus = tf.config.list_physical_devices('GPU')
print('TensorFlow:', tf.__version__)
print('GPU:', gpus or 'NOT FOUND — enable a GPU runtime')
if gpus:
    keras.mixed_precision.set_global_policy('mixed_float16')
print('Compute policy:', keras.mixed_precision.global_policy())

## 1. Kaggle authentication
Recommended: click **Generate New Token** at `https://www.kaggle.com/settings/api`, then add it to Colab **Secrets** as `KAGGLE_API_TOKEN`. The fallback supports legacy `KAGGLE_USERNAME` + `KAGGLE_KEY` or an uploaded `kaggle.json`. You must join the competition and accept its rules before downloading.

In [ ]:
def configure_kaggle():
    try:
        from google.colab import userdata
        token = userdata.get('KAGGLE_API_TOKEN')
        if token:
            os.environ['KAGGLE_API_TOKEN'] = token
            print('Using modern Kaggle API token from Colab Secrets.')
            return
        username = userdata.get('KAGGLE_USERNAME')
        key = userdata.get('KAGGLE_KEY')
        if username and key:
            os.environ['KAGGLE_USERNAME'] = username
            os.environ['KAGGLE_KEY'] = key
            print('Using Kaggle credentials from Colab Secrets.')
            return
    except Exception:
        pass

    from google.colab import files
    print('Upload kaggle.json from Kaggle Settings → API → Create New Token')
    uploaded = files.upload()
    if 'kaggle.json' not in uploaded:
        raise RuntimeError('kaggle.json was not uploaded.')
    kaggle_dir = Path.home() / '.kaggle'
    kaggle_dir.mkdir(exist_ok=True)
    credential_path = kaggle_dir / 'kaggle.json'
    credential_path.write_bytes(uploaded['kaggle.json'])
    credential_path.chmod(0o600)
    print('Kaggle credentials configured.')

configure_kaggle()

In [ ]:
COMPETITION = 'cassava-leaf-disease-classification'
ROOT = Path('/content/agrovision_cassava')
RAW = ROOT / 'raw'
OUTPUT = ROOT / 'outputs'
RAW.mkdir(parents=True, exist_ok=True)
OUTPUT.mkdir(parents=True, exist_ok=True)

download = subprocess.run(
    ['kaggle', 'competitions', 'download', '-c', COMPETITION, '-p', str(RAW), '--force'],
    text=True, capture_output=True
)
print(download.stdout[-1000:])
if download.returncode != 0:
    print(download.stderr[-2000:])
    raise RuntimeError('Kaggle download failed. Join the competition, accept its rules, and verify your API credentials.')

archives = list(RAW.glob('*.zip'))
if not archives:
    raise FileNotFoundError('Downloaded archive was not found.')
for archive in archives:
    shutil.unpack_archive(str(archive), str(RAW))
print('Extracted to', RAW)

## 2. Dataset audit and frozen split
The split manifest is saved before training. Exact duplicate files are removed before splitting to reduce leakage. The source dataset has no farm/plant identifiers, so a future Thai cohort must be grouped by farm and plant during collection.

In [ ]:
csv_candidates = list(RAW.rglob('train.csv'))
image_dir_candidates = [p for p in RAW.rglob('train_images') if p.is_dir()]
if not csv_candidates or not image_dir_candidates:
    raise FileNotFoundError('Expected train.csv and train_images were not found.')
CSV_PATH = csv_candidates[0]
IMAGE_DIR = image_dir_candidates[0]
LABEL_PATH = next(iter(RAW.rglob('label_num_to_disease_map.json')), None)

df = pd.read_csv(CSV_PATH)
required = {'image_id', 'label'}
if not required.issubset(df.columns):
    raise ValueError(f'train.csv must contain {required}; got {set(df.columns)}')

if LABEL_PATH:
    raw_labels = json.loads(LABEL_PATH.read_text())
    LABELS = {int(k): str(v) for k, v in raw_labels.items()}
else:
    LABELS = {
        0: 'Cassava Bacterial Blight (CBB)',
        1: 'Cassava Brown Streak Disease (CBSD)',
        2: 'Cassava Green Mottle (CGM)',
        3: 'Cassava Mosaic Disease (CMD)',
        4: 'Healthy',
    }
df['path'] = df['image_id'].map(lambda name: str(IMAGE_DIR / name))
df['exists'] = df['path'].map(lambda p: Path(p).exists())
if not df['exists'].all():
    raise FileNotFoundError(f"Missing {int((~df['exists']).sum())} images.")
print('Rows:', len(df), '| Classes:', LABELS)
display(df['label'].value_counts().sort_index().rename(index=LABELS).to_frame('images'))

In [ ]:
def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b''):
            digest.update(chunk)
    return digest.hexdigest()

started = time.time()
df['sha256'] = [sha256_file(path) for path in df['path']]
duplicate_mask = df.duplicated('sha256', keep='first')
conflicting = df.groupby('sha256')['label'].nunique().gt(1).sum()
print(f'Exact duplicates removed: {int(duplicate_mask.sum())}; conflicting-label hash groups: {int(conflicting)}; audit: {time.time()-started:.1f}s')
if conflicting:
    raise ValueError('Identical image bytes have conflicting labels. Review the dataset before training.')
clean = df.loc[~duplicate_mask].reset_index(drop=True)

train_df, temp_df = train_test_split(clean, test_size=0.30, stratify=clean['label'], random_state=SEED)
val_df, test_df = train_test_split(temp_df, test_size=0.50, stratify=temp_df['label'], random_state=SEED)
train_df = train_df.assign(split='train')
val_df = val_df.assign(split='val')
test_df = test_df.assign(split='test')
manifest = pd.concat([train_df, val_df, test_df]).sort_values(['split', 'image_id'])
manifest[['image_id', 'label', 'sha256', 'split']].to_csv(OUTPUT / 'split_manifest.csv', index=False)

split_table = pd.crosstab(manifest['split'], manifest['label']).rename(columns=LABELS)
display(split_table)
assert set(train_df.sha256).isdisjoint(val_df.sha256)
assert set(train_df.sha256).isdisjoint(test_df.sha256)
assert set(val_df.sha256).isdisjoint(test_df.sha256)

In [ ]:
fig, axes = plt.subplots(5, 4, figsize=(12, 15))
for label, row_axes in zip(sorted(LABELS), axes):
    samples = train_df[train_df.label == label].sample(4, random_state=SEED)
    for (_, row), ax in zip(samples.iterrows(), row_axes):
        ax.imshow(keras.utils.load_img(row.path))
        ax.set_title(LABELS[label], fontsize=9)
        ax.axis('off')
plt.suptitle('Training samples — visually inspect labels before continuing', y=1.01)
plt.tight_layout()

## 3. Input pipeline and model
Augmentation is applied only to training images. EfficientNetV2 includes input rescaling internally, so pixel values remain in the 0–255 range.

In [ ]:
IMG_SIZE = 224  # Must match the current AgroVision backend input contract
BATCH_SIZE = 32 if gpus else 8
NUM_CLASSES = len(LABELS)
AUTOTUNE = tf.data.AUTOTUNE

augment = keras.Sequential([
    keras.layers.RandomFlip('horizontal'),
    keras.layers.RandomRotation(0.08, fill_mode='reflect'),
    keras.layers.RandomZoom(0.12, fill_mode='reflect'),
    keras.layers.RandomTranslation(0.06, 0.06, fill_mode='reflect'),
    keras.layers.RandomContrast(0.15),
], name='field_augmentation')

def decode(path, label):
    image = tf.io.read_file(path)
    image = tf.io.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, [IMG_SIZE, IMG_SIZE], antialias=True)
    return tf.cast(image, tf.float32), tf.cast(label, tf.int32)

def make_dataset(frame, training=False):
    ds = tf.data.Dataset.from_tensor_slices((frame.path.values, frame.label.values))
    if training:
        ds = ds.shuffle(min(len(frame), 10000), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(decode, num_parallel_calls=AUTOTUNE)
    ds = ds.batch(BATCH_SIZE)
    if training:
        ds = ds.map(lambda x, y: (augment(x, training=True), y), num_parallel_calls=AUTOTUNE)
    return ds.prefetch(AUTOTUNE)

train_ds = make_dataset(train_df, training=True)
val_ds = make_dataset(val_df)
test_ds = make_dataset(test_df)

weights = compute_class_weight('balanced', classes=np.arange(NUM_CLASSES), y=train_df.label.values)
CLASS_WEIGHTS = {int(i): float(w) for i, w in enumerate(weights)}
print('Class weights:', CLASS_WEIGHTS)

In [ ]:
def build_model():
    inputs = keras.Input((IMG_SIZE, IMG_SIZE, 3), name='image')
    backbone = keras.applications.EfficientNetV2B0(
        include_top=False, weights='imagenet', input_tensor=inputs, pooling='avg'
    )
    backbone.trainable = False
    x = keras.layers.Dropout(0.30)(backbone.output)
    outputs = keras.layers.Dense(NUM_CLASSES, activation='softmax', dtype='float32', name='class_probabilities')(x)
    return keras.Model(inputs, outputs, name='agrovision_cassava_effnetv2b0'), backbone

def compile_model(model, learning_rate):
    model.compile(
        optimizer=keras.optimizers.AdamW(learning_rate=learning_rate, weight_decay=1e-5),
        loss=keras.losses.SparseCategoricalCrossentropy(),
        metrics=[
            keras.metrics.SparseCategoricalAccuracy(name='accuracy'),
            keras.metrics.SparseTopKCategoricalAccuracy(k=2, name='top2_accuracy'),
        ],
    )

model, backbone = build_model()
compile_model(model, 3e-4)
model.summary()

## 4. Stage 1 — train classification head
The maximum epoch count is only a ceiling. Early stopping restores the checkpoint with the lowest validation loss.

In [ ]:
BEST_HEAD = OUTPUT / 'best_head.keras'
head_callbacks = [
    keras.callbacks.ModelCheckpoint(BEST_HEAD, monitor='val_loss', save_best_only=True, verbose=1),
    keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True, verbose=1),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=2, min_lr=1e-6, verbose=1),
    keras.callbacks.CSVLogger(OUTPUT / 'training_head.csv'),
]
history_head = model.fit(
    train_ds, validation_data=val_ds, epochs=10, class_weight=CLASS_WEIGHTS, callbacks=head_callbacks
)

## 5. Stage 2 — controlled fine-tuning
Reload the best head checkpoint, unfreeze only the final 30% of the backbone, keep BatchNorm frozen, and use a much smaller learning rate.

In [ ]:
model = keras.models.load_model(BEST_HEAD)
backbone = next(layer for layer in model.layers if isinstance(layer, keras.Model) and 'efficientnet' in layer.name.lower())
backbone.trainable = True
freeze_until = int(len(backbone.layers) * 0.70)
for index, layer in enumerate(backbone.layers):
    layer.trainable = index >= freeze_until and not isinstance(layer, keras.layers.BatchNormalization)

trainable = sum(int(layer.trainable) for layer in backbone.layers)
print(f'Backbone layers trainable: {trainable}/{len(backbone.layers)}')
compile_model(model, 1e-5)

BEST_FINE = OUTPUT / 'cassava_best_model.keras'
fine_callbacks = [
    keras.callbacks.ModelCheckpoint(BEST_FINE, monitor='val_loss', save_best_only=True, verbose=1),
    keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.25, patience=2, min_lr=2e-7, verbose=1),
    keras.callbacks.CSVLogger(OUTPUT / 'training_finetune.csv'),
]
history_fine = model.fit(
    train_ds, validation_data=val_ds, epochs=30, class_weight=CLASS_WEIGHTS, callbacks=fine_callbacks
)

In [ ]:
def plot_history(histories):
    keys = ['loss', 'accuracy', 'top2_accuracy']
    fig, axes = plt.subplots(1, 3, figsize=(17, 4))
    for ax, key in zip(axes, keys):
        offset = 0
        for name, history in histories:
            n = len(history.history[key])
            epochs = np.arange(offset + 1, offset + n + 1)
            ax.plot(epochs, history.history[key], label=f'{name} train')
            ax.plot(epochs, history.history['val_' + key], '--', label=f'{name} val')
            offset += n
        ax.set_title(key); ax.set_xlabel('epoch'); ax.grid(alpha=.2); ax.legend(fontsize=8)
    plt.tight_layout()
    plt.savefig(OUTPUT / 'training_curves.png', dpi=180, bbox_inches='tight')

plot_history([('head', history_head), ('fine', history_fine)])

## 6. Calibration and frozen test evaluation
Temperature is selected on validation predictions only. The test set is evaluated once after all model and threshold decisions are frozen.

In [ ]:
model = keras.models.load_model(BEST_FINE)
val_probs = model.predict(val_ds, verbose=1)
test_probs = model.predict(test_ds, verbose=1)
val_true = val_df.label.to_numpy()
test_true = test_df.label.to_numpy()

def apply_temperature(probs, temperature):
    logits = np.log(np.clip(probs, 1e-7, 1.0)) / temperature
    logits -= logits.max(axis=1, keepdims=True)
    exp = np.exp(logits)
    return exp / exp.sum(axis=1, keepdims=True)

def nll_for_temperature(temperature):
    calibrated = apply_temperature(val_probs, temperature)
    return -np.log(np.clip(calibrated[np.arange(len(val_true)), val_true], 1e-9, 1)).mean()

temperature_result = minimize_scalar(nll_for_temperature, bounds=(0.25, 5.0), method='bounded')
TEMPERATURE = float(temperature_result.x)
test_probs_cal = apply_temperature(test_probs, TEMPERATURE)
print('Validation-fitted temperature:', round(TEMPERATURE, 4))

In [ ]:
def expected_calibration_error(y_true, probs, bins=15):
    confidence = probs.max(axis=1)
    prediction = probs.argmax(axis=1)
    correct = prediction == y_true
    edges = np.linspace(0, 1, bins + 1)
    ece = 0.0
    for lower, upper in zip(edges[:-1], edges[1:]):
        mask = (confidence > lower) & (confidence <= upper)
        if mask.any():
            ece += mask.mean() * abs(correct[mask].mean() - confidence[mask].mean())
    return float(ece)

def selective_metrics(y_true, probs, top1_min=.55, margin_min=.15, entropy_max=.65):
    order = np.sort(probs, axis=1)
    top1 = order[:, -1]
    margin = order[:, -1] - order[:, -2]
    entropy = -(probs * np.log(np.clip(probs, 1e-9, 1))).sum(axis=1) / np.log(probs.shape[1])
    accepted = (top1 >= top1_min) & (margin >= margin_min) & (entropy <= entropy_max)
    predictions = probs.argmax(axis=1)
    return {
        'coverage': float(accepted.mean()),
        'accepted_count': int(accepted.sum()),
        'accuracy_on_accepted': float((predictions[accepted] == y_true[accepted]).mean()) if accepted.any() else None,
        'accepted_mask': accepted,
    }

def bootstrap_ci(y_true, y_pred, metric, iterations=2000, seed=SEED):
    rng = np.random.default_rng(seed)
    values = []
    for _ in range(iterations):
        idx = rng.integers(0, len(y_true), len(y_true))
        values.append(metric(y_true[idx], y_pred[idx]))
    return [float(x) for x in np.percentile(values, [2.5, 97.5])]

test_pred = test_probs_cal.argmax(axis=1)
selective = selective_metrics(test_true, test_probs_cal)
report = classification_report(test_true, test_pred, target_names=[LABELS[i] for i in range(NUM_CLASSES)], output_dict=True, zero_division=0)
metrics = {
    'dataset': COMPETITION,
    'evaluation_scope': 'internal held-out African dataset; not Thai external validation',
    'samples': int(len(test_true)),
    'accuracy': float(accuracy_score(test_true, test_pred)),
    'accuracy_95ci': bootstrap_ci(test_true, test_pred, accuracy_score),
    'balanced_accuracy': float(balanced_accuracy_score(test_true, test_pred)),
    'macro_f1': float(f1_score(test_true, test_pred, average='macro')),
    'macro_f1_95ci': bootstrap_ci(test_true, test_pred, lambda y, p: f1_score(y, p, average='macro', zero_division=0)),
    'top2_accuracy': float(np.mean([truth in np.argsort(row)[-2:] for truth, row in zip(test_true, test_probs_cal)])),
    'ece_before': expected_calibration_error(test_true, test_probs),
    'ece_after': expected_calibration_error(test_true, test_probs_cal),
    'temperature': TEMPERATURE,
    'selective_prediction': {k: v for k, v in selective.items() if k != 'accepted_mask'},
    'per_class': report,
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
}
print(json.dumps({k: v for k, v in metrics.items() if k != 'per_class'}, indent=2))
(OUTPUT / 'evaluation_metrics.json').write_text(json.dumps(metrics, indent=2), encoding='utf-8')

In [ ]:
cm = confusion_matrix(test_true, test_pred, labels=np.arange(NUM_CLASSES))
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', xticklabels=[LABELS[i] for i in range(NUM_CLASSES)], yticklabels=[LABELS[i] for i in range(NUM_CLASSES)], ax=axes[0])
axes[0].set_title('Confusion matrix — counts'); axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('True')
cm_norm = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues', vmin=0, vmax=1, xticklabels=[LABELS[i] for i in range(NUM_CLASSES)], yticklabels=[LABELS[i] for i in range(NUM_CLASSES)], ax=axes[1])
axes[1].set_title('Confusion matrix — row normalized'); axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('True')
for ax in axes:
    ax.tick_params(axis='x', rotation=35); ax.tick_params(axis='y', rotation=0)
plt.tight_layout()
plt.savefig(OUTPUT / 'confusion_matrix.png', dpi=200, bbox_inches='tight')

In [ ]:
wrong = np.where(test_pred != test_true)[0]
if len(wrong):
    hardest = wrong[np.argsort(test_probs_cal[wrong].max(axis=1))[::-1][:16]]
    fig, axes = plt.subplots(4, 4, figsize=(14, 14))
    for idx, ax in zip(hardest, axes.flat):
        row = test_df.iloc[idx]
        ax.imshow(keras.utils.load_img(row.path))
        ax.set_title(f"True: {LABELS[int(test_true[idx])]}\nPred: {LABELS[int(test_pred[idx])]} ({test_probs_cal[idx].max():.1%})", fontsize=8)
        ax.axis('off')
    plt.suptitle('Most confident test errors — inspect for label noise and failure modes', y=.92)
    plt.tight_layout()
    plt.savefig(OUTPUT / 'confident_errors.png', dpi=180, bbox_inches='tight')
else:
    print('No test errors found.')

## 7. Export competition artifacts
The Keras model, float16 TFLite model, labels, calibration temperature, split manifest, plots, environment versions, and metrics are packaged together.

In [ ]:
final_model = keras.models.load_model(BEST_FINE)
FINAL_KERAS = OUTPUT / 'cassava_competition_model.keras'
final_model.save(FINAL_KERAS)

(OUTPUT / 'cassava_labels.json').write_text(json.dumps(LABELS, indent=2), encoding='utf-8')
(OUTPUT / 'calibration.json').write_text(json.dumps({
    'temperature': TEMPERATURE,
    'fit_split': 'validation',
    'selective_thresholds': {'top1_min': .55, 'margin_min': .15, 'normalized_entropy_max': .65}
}, indent=2), encoding='utf-8')

environment = {
    'python': sys.version,
    'tensorflow': tf.__version__,
    'numpy': np.__version__,
    'seed': SEED,
    'image_size': IMG_SIZE,
    'batch_size': BATCH_SIZE,
    'backbone': 'EfficientNetV2B0',
    'dataset_rows_after_exact_dedup': len(clean),
}
(OUTPUT / 'training_config.json').write_text(json.dumps(environment, indent=2), encoding='utf-8')

try:
    converter = tf.lite.TFLiteConverter.from_keras_model(final_model)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    converter.target_spec.supported_types = [tf.float16]
    (OUTPUT / 'cassava_competition_model_fp16.tflite').write_bytes(converter.convert())
    print('TFLite export complete.')
except Exception as exc:
    print('TFLite export failed; Keras model is still valid:', exc)

archive = shutil.make_archive('/content/agrovision_cassava_competition_artifacts', 'zip', OUTPUT)
print('Artifacts:', archive)
print('Keras SHA-256:', sha256_file(FINAL_KERAS))
print('Files:')
for path in sorted(OUTPUT.iterdir()):
    print(f'  {path.name}: {path.stat().st_size / 1_000_000:.2f} MB')

In [ ]:
# Optional: copy artifacts to Google Drive so they survive the Colab runtime.
from google.colab import drive
drive.mount('/content/drive')
DRIVE_OUTPUT = Path('/content/drive/MyDrive/AgroVision/cassava_competition_run')
if DRIVE_OUTPUT.exists():
    shutil.rmtree(DRIVE_OUTPUT)
shutil.copytree(OUTPUT, DRIVE_OUTPUT)
shutil.copy2('/content/agrovision_cassava_competition_artifacts.zip', DRIVE_OUTPUT.parent)
print('Saved to', DRIVE_OUTPUT)

## Competition reporting checklist

Report the frozen test accuracy, balanced accuracy, Macro-F1 with 95% bootstrap CI, per-class recall/F1, ECE before/after calibration, selective accuracy and coverage, latency on the actual demo device, and the model/dataset hashes.

Do not claim Thai field accuracy until the exact exported model is evaluated on a separately collected, expert-labelled Thai cohort that was never used for training, threshold selection, or calibration.